# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Logistic Regression first, then Random Forest.**

My lane's question is a yes/no shape — "will this page decline?" — which the
`training-honest-models` skill maps directly to Logistic Regression → Random
Forest. I start with Logistic Regression because it's fully readable (I can
name exactly which signals push the prediction up or down), then check if
Random Forest earns its extra complexity by actually beating it on the same
metric my baseline (ML-07) was judged on: Precision@50. If the forest doesn't
meaningfully beat logistic regression, I keep the simpler model — complexity
must earn its place, not be assumed.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
import duckdb, os
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_march = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
fact_april = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
content = f"read_parquet('{REL}/dim_content.parquet')"

# Grouped by client — pages from the same client never cross the train/test split
from sklearn.model_selection import GroupShuffleSplit

**Split: client-grouped, not random.** 

Pages from the same client can share
patterns (same industry, same content strategy), so a random split risks
letting the model memorize client-specific quirks rather than learn general
signal — the same risk the lane guide and data-contract work already flagged.
I split on `client_hash_id` using `GroupShuffleSplit`, so no client's pages
appear in both train and test. This is the honest split for this question:
the real use case is scoring pages for clients the model may not have seen
much of before, not pages from a client it already trained on.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
# Features — same discipline as ML-04/ML-07: prior-window only, no product flags
features = con.sql(f"""
    SELECT f.content_hash_id, f.client_hash_id,
           SUM(f.gsc_impressions) as impressions_march,
           AVG(f.gsc_avg_position) as avg_position,
           DATEDIFF('day', c.content_created_date, DATE '2026-03-31') as content_age_days,
           SUM(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END) as days_with_ga4
    FROM {fact_march} f JOIN {content} c ON f.content_hash_id = c.content_hash_id
    GROUP BY 1, 2, c.content_created_date
""").df()

# Label — future window (April), same idea as the ML-04 leakage demo, but
# used ONLY as the label here, never as a feature
april = con.sql(f"SELECT content_hash_id, SUM(gsc_impressions) as impressions_april FROM {fact_april} GROUP BY 1").df()
data = features.merge(april, on="content_hash_id", how="left").dropna()
data["is_declining"] = (data["impressions_april"] < data["impressions_march"]).astype(int)

X = data[["impressions_march", "avg_position", "content_age_days", "days_with_ga4"]].fillna(0)
y = data["is_declining"]
groups = data["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

logreg = LogisticRegression(class_weight="balanced").fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced").fit(X_train, y_train)

logreg_scores = logreg.predict_proba(X_test)[:, 1]
rf_scores = rf.predict_proba(X_test)[:, 1]

# My Week-4 baseline (rule-based) score, computed the same way as w04_baseline_score.ipynb,
# but scored on this same test set for a fair comparison
test_data = data.iloc[test_idx]
baseline_score = (
    (test_data["content_age_days"] >= 180).astype(int) *
    (test_data["impressions_march"] >= 500).astype(int)
) * test_data["impressions_march"]

print(f"{'Method':<22}{'Precision@50':>15}")
print(f"{'baseline (rule)':<22}{precision_at_k(baseline_score, y_test, 50):>15.3f}")
print(f"{'logistic regression':<22}{precision_at_k(logreg_scores, y_test, 50):>15.3f}")
print(f"{'random forest':<22}{precision_at_k(rf_scores, y_test, 50):>15.3f}")
print(f"{'base rate':<22}{y_test.mean():>15.3f}")

Method                   Precision@50
baseline (rule)                 0.500
logistic regression             0.660
random forest                   0.700
base rate                       0.578


| Method               | Precision@50 |
|----------------------|-------------:|
| baseline (rule)      | 0.500        |
| logistic regression  | 0.660        |
| random forest        | 0.700        |
| base rate            | 0.578        |

**Honest observation before celebrating the model:** my Week-4 rule-based
baseline (0.500) actually scores *below* the base rate (0.578) on this test
set — meaning picking 50 random pages would have outperformed my hand-written
rule here. This isn't a flattering result for the baseline, but it's an
important one: it shows the rule's "stale AND visible" logic wasn't actually
capturing decline risk well on this particular client-grouped split, more
than half of all test pages are declining, and the rule's narrow threshold
missed most of them.

Both learned models clearly beat both the rule and the base rate: logistic
regression reaches 0.660 and random forest reaches 0.700 — roughly 35 of the
top 50 flagged pages are correct with random forest, versus about 25 with
the baseline. The gain from logistic regression to random forest (0.660 to
0.700) is real but modest, which is useful to know: most of the signal here
is already captured by a simple, fully readable model.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
import pandas as pd
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)

# 3 concrete wrong cases
test_data = test_data.copy()
test_data["rf_score"] = rf_scores
test_data["actual"] = y_test.values
wrong = test_data[(test_data["rf_score"] > 0.7) & (test_data["actual"] == 0)].head(3)
print(wrong[["content_hash_id","impressions_march","avg_position","content_age_days","rf_score"]])

avg_position         0.366448
content_age_days     0.286995
impressions_march    0.283692
days_with_ga4        0.062865
dtype: float64
              content_hash_id  impressions_march  avg_position  \
170  content_60d2848f0cb872d2             3388.0      2.709339   
173  content_d216acf1972f4810            10995.0      5.007187   
176  content_b56e1bf2d18c8a0d             6013.0      4.099451   

     content_age_days  rf_score  
170                54     0.760  
173                49     0.740  
176                49     0.835  


**Feature importances (random forest):**
`avg_position` (0.366) dominates, followed by `content_age_days` (0.287) and
`impressions_march` (0.284) close behind; `days_with_ga4` contributes very
little (0.063). This makes sense directionally — search position is the
single strongest predictor of whether a page keeps or loses traffic, more
than raw traffic volume or staleness alone.

**Three concrete wrong cases (confident false positives — model predicted
high decline risk, but the page did not actually decline):**

| content_hash_id | impressions_march | avg_position | content_age_days | rf_score |
|---|---:|---:|---:|---:|
| content_60d2848f0cb872d2 | 3,388 | 2.71 | 54 | 0.760 |
| content_d216acf1972f4810 | 10,995 | 5.01 | 49 | 0.740 |
| content_b56e1bf2d18c8a0d | 6,013 | 4.10 | 49 | 0.835 |

**What these three have in common:** all three are relatively young pages
(49-54 days old, far from the "stale" range my baseline rule flagged), with
strong search positions (2.7-5.0) and decent-to-high traffic. The model was
very confident (0.74-0.84) that these would decline, but they didn't. This
suggests the model may have learned a spurious pattern in this narrow
age band from the training split — content around 49-54 days old with
these traffic/position characteristics happened to decline often enough in
training to earn a high score, but that pattern didn't hold for this
specific test subgroup. It's a reminder that `content_age_days` (the
model's second-most important feature) may be encoding a client-specific or
time-specific quirk rather than a stable, generalizable signal — worth
flagging for closer inspection in the capstone rather than trusting outright.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.